# Leaf bounding-box visualisation

Draws the leaf boxes of a COCO annotation file on the images, to check the detection labels before
training (Section 4.1).

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
COCO_JSON = ROOT / "data" / "annotations" / "instances_default.json"   # COCO export of the leaf boxes
IMAGE_DIR = ROOT / "data" / "images"
OUT_DIR = ROOT / "outputs" / "bbox_visualized"
N_PREVIEW = 6

In [ ]:
coco = json.loads(COCO_JSON.read_text(encoding="utf-8"))
categories = {c["id"]: c["name"] for c in coco["categories"]}
boxes = defaultdict(list)
for ann in coco["annotations"]:
    boxes[ann["image_id"]].append(ann)

print(f"images: {len(coco['images'])}  boxes: {len(coco['annotations'])}  classes: {len(categories)}")

In [ ]:
PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180), (70, 240, 240)]

def draw(info):
    # COCO boxes are in the EXIF display orientation used by the labelling tool.
    img = ImageOps.exif_transpose(Image.open(IMAGE_DIR / info["file_name"])).convert("RGB")
    d = ImageDraw.Draw(img)
    width = max(2, img.width // 300)
    for ann in boxes.get(info["id"], []):
        x, y, w, h = ann["bbox"]
        color = PALETTE[ann["category_id"] % len(PALETTE)]
        d.rectangle([x, y, x + w, y + h], outline=color, width=width)
        d.text((x + 4, y + 4), categories[ann["category_id"]], fill=color)
    return img

OUT_DIR.mkdir(parents=True, exist_ok=True)
missing = 0
for info in coco["images"]:
    if not (IMAGE_DIR / info["file_name"]).exists():
        missing += 1
        continue
    draw(info).save(OUT_DIR / info["file_name"], quality=90)
print(f"saved to {OUT_DIR}  (missing images: {missing})")

In [ ]:
sample = random.sample([i for i in coco["images"] if (IMAGE_DIR / i["file_name"]).exists()], N_PREVIEW)
fig, axes = plt.subplots(2, N_PREVIEW // 2, figsize=(15, 9))
for ax, info in zip(axes.ravel(), sample):
    ax.imshow(draw(info))
    ax.set_title(info["file_name"], fontsize=8)
    ax.axis("off")
plt.tight_layout()